# Seaborn Phase 2: Single Variable Distributions (Risk Profiling)
### Credit Card Risk Analysis Project

Before you can look for relationships between variables, you need to understand each
risk factor on its own — its shape, its spread, and where its outliers hide. Seaborn's
distribution plots do this with far less code than the raw Matplotlib equivalents, and
add a few things Matplotlib doesn't give you for free (like a smooth density curve).

This notebook covers 3 topics:
3. **Histplot (with KDE)** — distribution shape, plus a smoothed density curve
4. **Boxplot** — a cleaner, built-in-grouping way to spot outliers
5. **Violinplot** — a boxplot/density-curve hybrid that shows exactly where data is thickest

**Format:** Each question has a `YOUR CODE HERE` cell to attempt first, followed by a
`Solution` cell. Try your own answer before peeking!

Run the setup cell below first — it deliberately includes a handful of applicants
claiming an annual income of $5,000,000, so you have a real outlier to hunt for.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_theme(style="whitegrid")

n = 600

housing_status = np.random.choice(['Rent', 'Own', 'Mortgage'], size=n, p=[0.35, 0.25, 0.40])

credit_score = np.clip(np.random.normal(650, 70, n), 300, 850)

annual_income = np.random.lognormal(mean=10.8, sigma=0.4, size=n)
# Inject a handful of implausible high-income outliers, exactly the kind you'd
# want a boxplot to catch before modeling
outlier_idx = np.random.choice(n, size=5, replace=False)
annual_income[outlier_idx] = 5_000_000

total_debt = annual_income * np.random.uniform(0.05, 0.55, size=n)
debt_to_income = np.clip((total_debt / annual_income) * 100, 0, 90)

raw_risk = (debt_to_income / 100) * 0.6 + ((850 - credit_score) / 550) * 0.6
default_probability = np.clip(raw_risk + np.random.normal(0, 0.08, n), 0.01, 0.95)
default = np.random.binomial(1, default_probability)

risk_tier = pd.cut(credit_score, bins=[300, 600, 700, 850], labels=['High', 'Medium', 'Low'])

df = pd.DataFrame({
    'Housing_Status': housing_status,
    'Credit_Score': credit_score,
    'Annual_Income': annual_income,
    'Debt_to_Income': debt_to_income,
    'Default_Probability': default_probability,
    'Default': default,
    'Risk_Tier': risk_tier
})

print(df.shape)
df.head()


---
## Section 3: The Histplot (with KDE)

`sns.histplot()` is Seaborn's histogram — same idea as `ax.hist()`, but with built-in
options for a smoothed density curve (KDE), grouping by category, and normalization,
all as simple keyword arguments.


**Q1.** Plot a basic histogram of `Credit_Score` using `sns.histplot(data=df, x='Credit_Score')`.

In [ ]:
# YOUR CODE HERE


**Solution 1**

In [ ]:
sns.histplot(data=df, x='Credit_Score')
plt.title("Credit Score Distribution")
plt.show()


**Q2.** Repeat Q1, adding `kde=True` to overlay a smooth density curve on top of the bars — this makes it much easier to answer "is this tightly clustered or spread out?" at a glance than bars alone.

In [ ]:
# YOUR CODE HERE


**Solution 2**

In [ ]:
sns.histplot(data=df, x='Credit_Score', kde=True)
plt.title("Credit Score Distribution with KDE")
plt.show()


**Q3.** Repeat Q2, controlling the bin count with `bins=40` for a finer view, and setting `color='steelblue'`.

In [ ]:
# YOUR CODE HERE


**Solution 3**

In [ ]:
sns.histplot(data=df, x='Credit_Score', kde=True, bins=40, color='steelblue')
plt.title("Credit Score Distribution (40 bins)")
plt.show()


**Q4.** Repeat Q2, but customize just the KDE line's appearance using `line_kws={'color': 'red', 'linewidth': 2}` — a dictionary of keyword arguments passed straight through to the underlying line plot.

In [ ]:
# YOUR CODE HERE


**Solution 4**

In [ ]:
sns.histplot(data=df, x='Credit_Score', kde=True, line_kws={'color': 'red', 'linewidth': 2})
plt.title("Credit Score Distribution (custom KDE line)")
plt.show()


**Q5.** Plot `Debt_to_Income` split by `hue='Risk_Tier'`, with `kde=True`. Pass `element='step'` so the overlapping histograms show as outlines rather than solid filled bars, which stays readable even with three overlapping groups.

In [ ]:
# YOUR CODE HERE


**Solution 5**

In [ ]:
sns.histplot(data=df, x='Debt_to_Income', hue='Risk_Tier', hue_order=['Low', 'Medium', 'High'],
             kde=True, element='step')
plt.title("Debt-to-Income Distribution by Risk Tier")
plt.show()


**Q6.** Repeat Q1's `Credit_Score` histogram, but set `stat='percent'` instead of the default raw count — now the y-axis shows what percentage of applicants fall in each bin, which is easier to communicate to a non-technical stakeholder than a raw count.

In [ ]:
# YOUR CODE HERE


**Solution 6**

In [ ]:
sns.histplot(data=df, x='Credit_Score', stat='percent')
plt.ylabel("Percent of Applicants")
plt.title("Credit Score Distribution (%)")
plt.show()


**Q7.** Use `sns.kdeplot(data=df, x='Debt_to_Income', hue='Default', fill=True)` — the standalone density-only plot (no bars at all), filled underneath, comparing the DTI distribution of applicants who defaulted (`Default=1`) against those who didn't (`Default=0`).

In [ ]:
# YOUR CODE HERE


**Solution 7**

In [ ]:
sns.kdeplot(data=df, x='Debt_to_Income', hue='Default', fill=True)
plt.title("Debt-to-Income Density: Defaulted vs. Paid")
plt.show()


**Q8.** Plot `Annual_Income` as a histogram with `kde=True`. Notice how the handful of $5,000,000 outliers we injected completely dominate the x-axis and flatten everything else. Now add `log_scale=True` to `sns.histplot()` — Seaborn's built-in shortcut for a log-scaled axis, sparing you the manual `np.logspace` bin-edge work from the Matplotlib phases.

In [ ]:
# YOUR CODE HERE


**Solution 8**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(data=df, x='Annual_Income', kde=True, ax=axes[0])
axes[0].set_title("Linear Scale (outliers dominate)")

sns.histplot(data=df, x='Annual_Income', kde=True, log_scale=True, ax=axes[1])
axes[1].set_title("Log Scale (log_scale=True)")

plt.show()


**Q9 (Capstone).** Build a fully styled distribution chart: `Credit_Score` histogram with `kde=True`, `hue='Risk_Tier'` (ordered Low/Medium/High), `element='step'`, `stat='density'`, a bold Matplotlib title, axis labels, and `sns.despine()`.

In [ ]:
# YOUR CODE HERE


**Solution 9**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=df, x='Credit_Score', hue='Risk_Tier', hue_order=['Low', 'Medium', 'High'],
             kde=True, element='step', stat='density', ax=ax)
ax.set_title("Credit Score Density by Risk Tier", fontsize=13, fontweight='bold')
ax.set_xlabel("Credit Score")
ax.set_ylabel("Density")
sns.despine(ax=ax)
plt.show()


> **Checkpoint — Section 3:** `sns.histplot(kde=True)` gets you bars plus a smoothed
> density curve in one call. `hue=` handles group comparisons Matplotlib would need a
> manual loop for, `stat=` switches between count/density/percent, and `log_scale=True`
> is a one-keyword shortcut for what took manual bin-edge work in Matplotlib.


---
## Section 4: The Boxplot

`sns.boxplot()` does the same job as Matplotlib's `ax.boxplot()`, but grouping by a
category and coloring by another are both just keyword arguments — no manual filtering
into separate arrays required.


**Q10.** Plot a boxplot of `Annual_Income` using `sns.boxplot(data=df, y='Annual_Income')`. The $5,000,000 outliers should appear as individual points far above the box.

In [ ]:
# YOUR CODE HERE


**Solution 10**

In [ ]:
sns.boxplot(data=df, y='Annual_Income')
plt.title("Annual Income: Outlier Check")
plt.show()


**Q11.** Plot `sns.boxplot(data=df, x='Risk_Tier', y='Debt_to_Income', order=['Low', 'Medium', 'High'])` — a grouped boxplot with no manual filtering into separate arrays needed, unlike the equivalent Matplotlib code from earlier phases.

In [ ]:
# YOUR CODE HERE


**Solution 11**

In [ ]:
sns.boxplot(data=df, x='Risk_Tier', y='Debt_to_Income', order=['Low', 'Medium', 'High'])
plt.title("Debt-to-Income by Risk Tier")
plt.show()


**Q12.** Repeat Q10's `Annual_Income` boxplot twice side by side in a 1x2 subplot grid: left with default settings, right with `showfliers=False` (hides the outlier points and rescales the y-axis to the non-outlier data) — a quick way to see the "normal" spread once you already know the outliers are there.

In [ ]:
# YOUR CODE HERE


**Solution 12**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

sns.boxplot(data=df, y='Annual_Income', ax=axes[0])
axes[0].set_title("With Outliers")

sns.boxplot(data=df, y='Annual_Income', showfliers=False, ax=axes[1])
axes[1].set_title("Outliers Hidden (showfliers=False)")

plt.show()


**Q13.** Plot `sns.boxplot(data=df, x='Risk_Tier', y='Debt_to_Income', hue='Housing_Status', order=['Low', 'Medium', 'High'])` — a second grouping variable via `hue`, giving you sub-boxes within each Risk Tier for every Housing Status.

In [ ]:
# YOUR CODE HERE


**Solution 13**

In [ ]:
sns.boxplot(data=df, x='Risk_Tier', y='Debt_to_Income', hue='Housing_Status',
            order=['Low', 'Medium', 'High'])
plt.title("Debt-to-Income by Risk Tier and Housing Status")
plt.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
plt.show()


**Q14.** Plot the same Risk-Tier-grouped `Debt_to_Income` boxplot, but horizontally by swapping which variable goes to `x` vs `y`: `sns.boxplot(data=df, x='Debt_to_Income', y='Risk_Tier', order=['Low', 'Medium', 'High'])`.

In [ ]:
# YOUR CODE HERE


**Solution 14**

In [ ]:
sns.boxplot(data=df, x='Debt_to_Income', y='Risk_Tier', order=['Low', 'Medium', 'High'])
plt.title("Debt-to-Income by Risk Tier (horizontal)")
plt.show()


**Q15.** Repeat Q11's Risk-Tier boxplot, passing a custom palette: `palette={'Low': 'seagreen', 'Medium': 'orange', 'High': 'tomato'}` — a dictionary mapping category values directly to colors, so the mapping stays correct no matter what order the categories are drawn in.

In [ ]:
# YOUR CODE HERE


**Solution 15**

In [ ]:
risk_palette = {'Low': 'seagreen', 'Medium': 'orange', 'High': 'tomato'}

sns.boxplot(data=df, x='Risk_Tier', y='Debt_to_Income', hue='Risk_Tier',
            order=['Low', 'Medium', 'High'], palette=risk_palette, legend=False)
plt.title("Debt-to-Income by Risk Tier")
plt.show()


**Q16.** Plot the `Annual_Income` boxplot (with its $5,000,000 outliers included), then call plain Matplotlib's `plt.yscale('log')` on top of it — combining a Seaborn boxplot with a Matplotlib log-scaled axis so the box itself stays readable even with an extreme outlier present.

In [ ]:
# YOUR CODE HERE


**Solution 16**

In [ ]:
sns.boxplot(data=df, y='Annual_Income')
plt.yscale('log')
plt.title("Annual Income (log-scaled y-axis)")
plt.show()


**Q17.** Programmatically confirm what Q10's boxplot showed visually: filter `df` for rows where `Annual_Income > 1_000_000` and print the `Housing_Status`, `Annual_Income`, and `Risk_Tier` columns for those rows.

In [ ]:
# YOUR CODE HERE


**Solution 17**

In [ ]:
extreme_income = df[df['Annual_Income'] > 1_000_000]
print(extreme_income[['Housing_Status', 'Annual_Income', 'Risk_Tier']])


**Q18 (Capstone).** Build the final version: `sns.boxplot(data=df, x='Risk_Tier', y='Annual_Income', order=['Low', 'Medium', 'High'])` with the custom `risk_palette` from Q15, `plt.yscale('log')` so the $5,000,000ers don't flatten everything else, a bold title, and axis labels.

In [ ]:
# YOUR CODE HERE


**Solution 18**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=df, x='Risk_Tier', y='Annual_Income', hue='Risk_Tier',
            order=['Low', 'Medium', 'High'], palette=risk_palette, legend=False, ax=ax)
ax.set_yscale('log')
ax.set_title("Annual Income by Risk Tier (log scale)", fontsize=13, fontweight='bold')
ax.set_xlabel("Risk Tier")
ax.set_ylabel("Annual Income (log scale)")
plt.show()


> **Checkpoint — Section 4:** `sns.boxplot()` matches Matplotlib's outlier-spotting power
> but skips the manual array-filtering — `x=`, `hue=`, and `palette=` (as a dict) handle
> grouping and coloring directly from a DataFrame. `showfliers=False` and a log-scaled
> y-axis are both ways to keep a chart readable once extreme outliers are already known
> about.


---
## Section 5: The Violinplot

A violin plot is a boxplot and a density curve combined: the width at any point on the
"violin" shows how much data sits at that value, while the shape as a whole still lets
you read quartiles the way a boxplot does.


**Q19.** Plot a basic violin plot of `Credit_Score` using `sns.violinplot(data=df, y='Credit_Score')`.

In [ ]:
# YOUR CODE HERE


**Solution 19**

In [ ]:
sns.violinplot(data=df, y='Credit_Score')
plt.title("Credit Score Distribution")
plt.show()


**Q20.** Plot `sns.violinplot(data=df, x='Risk_Tier', y='Credit_Score', order=['Low', 'Medium', 'High'])` — one violin per Risk Tier, so you can compare both spread and density shape across groups at once.

In [ ]:
# YOUR CODE HERE


**Solution 20**

In [ ]:
sns.violinplot(data=df, x='Risk_Tier', y='Credit_Score', order=['Low', 'Medium', 'High'])
plt.title("Credit Score by Risk Tier")
plt.show()


**Q21.** Plot `Debt_to_Income` by `Risk_Tier`, but pass `hue='Default'` and `split=True` — with a binary `hue` (`Default` is 0 or 1), `split=True` draws each half of the violin from one hue group, letting you compare defaulted vs. paid applicants directly within each Risk Tier.

In [ ]:
# YOUR CODE HERE


**Solution 21**

In [ ]:
sns.violinplot(data=df, x='Risk_Tier', y='Debt_to_Income', hue='Default',
               order=['Low', 'Medium', 'High'], split=True)
plt.title("Debt-to-Income by Risk Tier, Split by Default Status")
plt.show()


**Q22.** Compare the `inner` parameter, which controls what's drawn inside the violin. Create a 1x3 subplot grid, and plot the same `Credit_Score`-by-`Risk_Tier` violin plot three times with `inner='box'`, `inner='quartile'`, and `inner='stick'` respectively, titling each panel with the `inner` value used.

In [ ]:
# YOUR CODE HERE


**Solution 22**

In [ ]:
inner_options = ['box', 'quartile', 'stick']

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, inner in zip(axes, inner_options):
    sns.violinplot(data=df, x='Risk_Tier', y='Credit_Score', order=['Low', 'Medium', 'High'],
                    inner=inner, ax=ax)
    ax.set_title(f"inner='{inner}'")

plt.show()


**Q23.** Create a 1x2 subplot grid comparing the same data two ways: left panel a boxplot of `Debt_to_Income` by `Risk_Tier`, right panel a violin plot of the same. This side-by-side is the clearest way to see exactly what a violin plot adds over a boxplot — the density shape underneath the quartile information.

In [ ]:
# YOUR CODE HERE


**Solution 23**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

sns.boxplot(data=df, x='Risk_Tier', y='Debt_to_Income', order=['Low', 'Medium', 'High'], ax=axes[0])
axes[0].set_title("Boxplot")

sns.violinplot(data=df, x='Risk_Tier', y='Debt_to_Income', order=['Low', 'Medium', 'High'], ax=axes[1])
axes[1].set_title("Violin Plot")

fig.suptitle("Debt-to-Income by Risk Tier: Two Views of the Same Data")
plt.show()


**Q24.** Plot `sns.violinplot(data=df, x='Risk_Tier', y='Credit_Score', order=['Low', 'Medium', 'High'], palette=risk_palette)` reusing the custom palette from Section 4, with `inner='quartile'`.

In [ ]:
# YOUR CODE HERE


**Solution 24**

In [ ]:
sns.violinplot(data=df, x='Risk_Tier', y='Credit_Score', hue='Risk_Tier',
               order=['Low', 'Medium', 'High'], palette=risk_palette, legend=False, inner='quartile')
plt.title("Credit Score by Risk Tier")
plt.show()


**Q25.** Look back at Q21's split violin plot of `Debt_to_Income` by `Risk_Tier` and `Default`. Answer in a markdown-style comment or print statement: within the `'High'` risk tier, does the defaulted half of the violin appear noticeably wider at higher DTI values than the paid half? Verify your visual read by computing `df[df['Risk_Tier'] == 'High'].groupby('Default')['Debt_to_Income'].mean()`.

In [ ]:
# YOUR CODE HERE


**Solution 25**

In [ ]:
high_tier_dti = df[df['Risk_Tier'] == 'High'].groupby('Default')['Debt_to_Income'].mean()
print(high_tier_dti)
# If the Default=1 mean is noticeably higher than Default=0, that confirms what the
# split violin's wider upper half within 'High' suggested visually.


**Q26 (Capstone).** Build the final "risk profiling" chart: a split violin plot of `Debt_to_Income` by `Risk_Tier` (ordered Low/Medium/High) and `Default` (`split=True`), with `inner='quartile'`, a `palette` of your choice for the two `Default` values (e.g. `{0: 'seagreen', 1: 'tomato'}`), a bold title, axis labels, and `sns.despine()`.

In [ ]:
# YOUR CODE HERE


**Solution 26**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.violinplot(data=df, x='Risk_Tier', y='Debt_to_Income', hue='Default',
               order=['Low', 'Medium', 'High'], split=True, inner='quartile',
               palette={0: 'seagreen', 1: 'tomato'}, ax=ax)
ax.set_title("Debt-to-Income by Risk Tier, Split by Default Status", fontsize=13, fontweight='bold')
ax.set_xlabel("Risk Tier")
ax.set_ylabel("Debt-to-Income Ratio (%)")
ax.legend(title='Default', labels=['Paid', 'Defaulted'])
sns.despine(ax=ax)
plt.show()


---
## Checkpoint: Seaborn Phase 2 Complete

You've covered:
- **Histplot with KDE** — `sns.histplot(kde=True)`, `hue=` group comparisons,
  `stat='percent'`/`'density'`, `element='step'` for overlapping groups, standalone
  `sns.kdeplot()`, and `log_scale=True` as a one-keyword alternative to manual log bins
- **Boxplot** — grouping and coloring directly from a DataFrame via `x=`/`hue=`/`palette=`,
  `showfliers=False`, and combining with a Matplotlib log-scaled axis for extreme outliers
- **Violinplot** — density shape plus quartile info in one chart, `split=True` for
  comparing a binary group within each category, and the `inner=` options for what's
  drawn inside

Together these three chart types are your single-variable risk profiling toolkit — shape
(histplot/KDE), outliers (boxplot), and shape-plus-outliers-plus-group-comparison all at
once (violinplot).

**Next up:** likely Seaborn's tools for relationships between *two* variables —
`sns.scatterplot()` with regression lines (`sns.regplot()`/`sns.lmplot()`), and
`sns.pairplot()` for viewing every pairwise relationship at once. Let me know when
you're ready!
